In [1]:
import pickle, os
import requests as reqlib

import pandas as pd
import numpy as np
import cma

In [2]:
reference_path = "../../results/transit/reference.parquet"
# routing_endpoint = "http://sma.univ-eiffel.fr:18054//router/transit"
routing_endpoint = "http://localhost:8054/router/transit"

output_path = "../../results/transit/calibration.p"

# Objective is observation_based or distribution_based
objective = "distribution_based"

In [3]:
#if "papermill" in locals():
#    survey_path = papermill.input["survey"]
#    spatial_path = papermill.input["spatial"]

#    routing_endpoint = papermill.params["routing_endpoint"]
#    selected_objective = "individual"

#    progress_path = papermill.output["progress"]
#    output_path = papermill.output["parameters"]

In [4]:
# Load reference data
df_reference = pd.read_parquet(reference_path)

# df_reference = df_reference.iloc[:1000] # For testing

In [5]:
# Identify modes and maximum transfers
modes = [c.replace("legs_", "") for c in df_reference.columns if c.startswith("legs_")]
maximum_transfers = df_reference["transfers"].max()

In [6]:
# Convert to requests
requests = []

for index, row in df_reference.iterrows():
    requests.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": row["departure_time"]
    })

In [7]:
# Prepare querying the routing server
def query_endpoint(requests, utilities):
    response = reqlib.post(routing_endpoint, json = {
        "batch": requests,
        "utilities": utilities
    })

    assert response.status_code == 200

    df_response = { 
        "request_index": [],
        "transfers": []
    }

    for mode in modes:
        df_response["legs_{}".format(mode)] = []

    for row in response.json():
        df_response["request_index"].append(row["request_index"])
        df_response["transfers"].append(np.minimum(row["transfers"], maximum_transfers))
        
        for mode in modes:
            if mode in row["vehicle_legs_by_mode"]:
                df_response["legs_{}".format(mode)].append(row["vehicle_legs_by_mode"][mode])
            else:
                df_response["legs_{}".format(mode)].append(0)
    
    return pd.DataFrame(df_response)

In [8]:
# We need to be careful about the request/response size, so we send individual batches
maximum_batch_size = 4000

def query_endpoint_batched(requests, utilities):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(requests):
        df_response.append(query_endpoint(
            requests[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size],
            utilities))
        
        batch_index += 1
    
    return pd.concat(df_response)

In [9]:
# Define calibration variables
variables = [
    { "name": "rail_u_h", "initial": -1.0 },
    { "name": "subway_u_h", "initial": -1.0, "fixed": True },
    { "name": "bus_u_h", "initial": -1.0 },
    { "name": "tram_u_h", "initial": -1.0 },
    { "name": "other_u_h", "copy": "bus_u_h" },
    { "name": "wait_u_h", "initial": -1.0 },
    { "name": "walk_u_h", "initial": -1.0 },
    { "name": "transfer_u", "initial": -1.0 }
]

In [10]:
# Extend with index information for CMA-ES evaluation
variables_map = { v["name"]: v for v in variables }

active_index = 0

# First find active variables
for variable in variables:
    if "fixed" in variable or "copy" in variable: 
        continue

    variables_map[variable["name"]] = variable
    
    variable["index"] = active_index
    variable["optimized"] = True
    active_index += 1

# Then treat fixed variables and those copying from others
for variable in variables:
    if "fixed" in variable:
        variable["index"] = None
    
    if "copy" in variable:
        assert not "initial" in variable
        variable["initial"] = variables_map[variable["copy"]]["initial"]
        variable["index"] = variables_map[variable["copy"]]["index"]

In [11]:
# Define the optimization objective
modes_weight = 1.0
transfers_weight = 1.0

def calculate_objective_observation_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    df_evaluation["offset"] = transfers_weight * np.abs(
        df_evaluation["transfers_reference"] - df_evaluation["transfers_evaluation"])

    for mode in modes:
        df_evaluation["offset"] += modes_weight * np.abs(
            df_evaluation["legs_{}_reference".format(mode)] - df_evaluation["legs_{}_evaluation".format(mode)]
        )

    return np.sum(df_evaluation["offset"] * df_evaluation["weight"]) / df_evaluation["weight"].sum()

def calculate_objective_distribution_based(df_evaluation):
    df_evaluation = pd.merge(df_reference, df_evaluation, on = "request_index", 
        suffixes = ["_reference", "_evaluation"])
    
    reference_mode_distribution = []
    evaluation_mode_distribution = []

    for mode in modes:
        reference_mode_distribution.append((df_evaluation["legs_{}_reference".format(mode)] * df_evaluation["weight"]).sum())
        evaluation_mode_distribution.append((df_evaluation["legs_{}_evaluation".format(mode)] * df_evaluation["weight"]).sum())

    reference_mode_distribution = np.array(reference_mode_distribution) / np.sum(reference_mode_distribution)
    evaluation_mode_distribution = np.array(evaluation_mode_distribution) / np.sum(evaluation_mode_distribution)

    reference_transfer_distribution = []
    evaluation_transfer_distribution = []

    for transfers in range(maximum_transfers + 1):
        f_reference = df_evaluation["transfers_reference"] == transfers
        reference_transfer_distribution.append(df_evaluation.loc[f_reference, "weight"].sum())

        f_evaluation = df_evaluation["transfers_evaluation"] == transfers
        evaluation_transfer_distribution.append(df_evaluation.loc[f_evaluation, "weight"].sum())

    reference_transfer_distribution = np.array(reference_transfer_distribution) / np.sum(reference_transfer_distribution)
    evaluation_transfer_distribution = np.array(evaluation_transfer_distribution) / np.sum(evaluation_transfer_distribution)

    mode_distribution_offset = np.abs(reference_mode_distribution - evaluation_mode_distribution)
    transfer_distribution_offset = np.abs(reference_transfer_distribution - evaluation_transfer_distribution)

    return transfers_weight * np.sum(transfer_distribution_offset) + modes_weight * np.sum(mode_distribution_offset)

In [12]:
# Prepare function to convert CMA-ES' candidate to utilities
def prepare_utilities(values):
    utilities = {}
    
    for variable in variables:
        if variable["index"] is not None:
            utilities[variable["name"]] = values[variable["index"]]
        else:
            utilities[variable["name"]] = variable["initial"]
    
    return utilities

In [13]:
# Prepare bounds and initial values
initial = []
bounds = [[], []]

for variable in variables:
    if "optimized" in variable:
        initial.append(variable["initial"])
        bounds[0].append(-np.inf)
        bounds[1].append(0.0)

In [14]:
# Test connection
assert len(query_endpoint(requests[:5], prepare_utilities(initial))) == 5

In [15]:
# Configure CMA-ES
seed = 1000
sigma = 0.5
iterations = 1000

options = cma.CMAOptions()
options.set("bounds", bounds)
options.set("seed", seed)

algorithm = cma.CMAEvolutionStrategy(initial, sigma, options)

# Load cached data for previous iterations
history = []

if os.path.exists(output_path):
    with open(output_path, "rb") as f:
        history = pickle.load(f)

        algorithm.feed_for_resume(
            [h["candidate"] for h in history[1:]], # first one is initial
            [h["objective"] for h in history[1:]]
        )

# Choose objective
if objective == "observation_based":
    calculate_objective = calculate_objective_observation_based
elif objective == "distribution_based":
    calculate_objective = calculate_objective_distribution_based
else:
    raise RuntimeError("Unknown objective")

# Perform a new batch of iterations
for iteration in range(iterations):
    initial_evaluation = len(history) == 0
    candidates = [initial]

    if not initial_evaluation:
        candidates = algorithm.ask()

    objectives = []

    for candidate in candidates:
        utilities = prepare_utilities(candidate)
        df_response = query_endpoint_batched(requests, utilities)
        objective = calculate_objective(df_response)

        objectives.append(objective)

        history.append({
            "candidate": candidate,
            "utilities": utilities,
            "objective": objective,
            "evaluation": df_response,
            "initial": initial_evaluation
        })

    if not initial_evaluation:
        algorithm.tell(candidates, objectives)
        algorithm.disp()

    # Save after a successful CMA-ES iteration
    with open(output_path, "wb+") as f:
        pickle.dump(history, f)

(4_w,9)-aCMA-ES (mu_w=2.8,w_1=49%) in dimension 6 (seed=1000, Fri Apr 11 15:35:20 2025)
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1      9 3.649804756012069e-01 1.0e+00 4.79e-01  4e-01  5e-01 1:30.1


C:\Users\lebescond\AppData\Local\Temp\ipykernel_12056\359305319.py:31: RuntimeWarning: invalid value encountered in divide
  evaluation_mode_distribution = np.array(evaluation_mode_distribution) / np.sum(evaluation_mode_distribution)


    2     18 3.513764565981969e-01 1.2e+00 5.12e-01  5e-01  6e-01 2:59.0


c:\Users\lebescond\AppData\Local\miniforge3\envs\choice_model\Lib\site-packages\cma\utilities\utils.py:347: UserWarning: function values with index [6]/[] are nan/None and will be set to the median value 0.7821865473600231 (time=Apr 11 15:38:33 2025 class=CMAEvolutionStrategy method=ask iteration=1)
  warnings.warn(msg + ' (time={}'.format(time.asctime()[4:]) +


    3     27 2.565431104772509e-01 1.5e+00 4.86e-01  4e-01  6e-01 4:23.6
    4     36 4.650240047093748e-01 1.5e+00 5.31e-01  5e-01  6e-01 5:50.3
    5     45 2.329144238319463e-01 1.4e+00 5.61e-01  5e-01  7e-01 7:21.7
    6     54 5.458518171391553e-01 1.5e+00 5.71e-01  5e-01  7e-01 9:01.9
    7     63 2.703377051913791e-01 1.6e+00 5.04e-01  4e-01  6e-01 10:45.9
    8     72 2.533015213850938e-01 1.8e+00 4.35e-01  3e-01  5e-01 12:18.7
    9     81 2.935791510483916e-01 1.9e+00 3.77e-01  3e-01  4e-01 13:59.7
   10     90 2.764493627123826e-01 1.9e+00 3.38e-01  2e-01  4e-01 15:44.1
   11     99 3.062208192560108e-01 1.8e+00 2.93e-01  2e-01  3e-01 17:26.6
   12    108 3.015201092687769e-01 1.9e+00 2.68e-01  2e-01  3e-01 19:07.9
   13    117 3.173788888166528e-01 2.0e+00 2.95e-01  2e-01  3e-01 20:47.6
   14    126 3.175557926795596e-01 1.7e+00 2.93e-01  2e-01  3e-01 22:23.2
   15    135 2.128909789979224e-01 1.8e+00 3.10e-01  2e-01  3e-01 24:04.0
   16    144 2.074699086760762e-01 2.1e+00

KeyboardInterrupt: 